# V-Modul 526 &mdash; Versuch 3
# Messen der Signale des MC4 Rezeptors: Luciferase-Reportergen-Assay

**Auswertung der Konzentrations-Wirkungs-Kurve (KWK) mit Python / JupyterLite**

---

### Worum geht es?

Der MC4 Rezeptor ist ein G<sub>s</sub>-gekoppelter GPCR. Nach Bindung von &alpha;-MSH aktiviert
G&alpha;<sub>s</sub> die Adenylylcyclase &rarr; cAMP &uarr; &rarr; PKA &rarr; CREB-P &rarr; Bindung an das
*cAMP response element* (CRE). Hinter dem CRE steht in diesem Versuch das Reportergen der
**Firefly-Luciferase**. Die gemessene Lumineszenz (Counts/s) ist damit ein Mass fuer die
G<sub>s</sub>-Signalaktivitaet des Rezeptors.

Verglichen werden drei Konditionen:

| Kondition | Bedeutung |
|---|---|
| **MC4R** | wildtypischer MC4 Rezeptor |
| **mut. MC4R** | MC4 Rezeptor mit der Adipositas-Mutation Ile194Phe |
| **pcDps** | Leervektor &ndash; Negativkontrolle |

### Was macht dieses Notebook?

1. Rohdaten (384-Well-Platte, Tecan Spark) direkt aus der Excel-Datei einlesen
2. Wells ueber das Pipettierschema (Skript S. 28, Abb. 7) den Konditionen zuordnen
3. Mittelwert und Standardabweichung der Triplikate berechnen
4. Normierung auf den **unstimulierten Leervektor** (pcDps, 0 M) &rarr; *x-fold of pcDps unstimulated*
5. Grafische Darstellung der KWK und der Kontrollen
6. Kurvenanpassung (*log(agonist) vs. response*, 3 Parameter) und Bestimmung der EC<sub>50</sub>-Werte

---
## 1. Module importieren

Alle benoetigten Pakete werden hier &ndash; und nur hier &ndash; geladen.

In **JupyterLite** laeuft Python im Browser (Pyodide). `numpy`, `pandas`, `scipy` und
`matplotlib` sind dort vorinstalliert, `openpyxl` (zum Lesen von `.xlsx`) muss einmalig
nachinstalliert werden. Die naechste Zelle erledigt das automatisch und funktioniert
gleichzeitig in einem normalen lokalen Jupyter.

In [ ]:
# --- Nachinstallation in JupyterLite (im lokalen Jupyter wird dieser Block uebersprungen) ---
import sys

IN_JUPYTERLITE = ("pyodide" in sys.modules) or (sys.platform == "emscripten")

if IN_JUPYTERLITE:
    try:
        import openpyxl                      # schon vorhanden?
    except ImportError:
        import piplite                       # JupyterLite-eigener Paketmanager
        await piplite.install("openpyxl")    # noqa: F704  (top-level await ist in JupyterLite erlaubt)

# --- Standardbibliothek ---
import re
import io
from pathlib import Path

# --- Numerik & Daten ---
import numpy as np
import pandas as pd

# --- Excel ---
import openpyxl

# --- Kurvenanpassung ---
from scipy.optimize import curve_fit

# --- Grafik ---
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

%matplotlib inline

# --- globale Darstellungseinstellungen ---
mpl.rcParams.update({
    "figure.dpi":        110,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
    "font.family":       "sans-serif",
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.labelsize":    10,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

pd.set_option("display.float_format", lambda v: f"{v:.4f}")

print("Umgebung :", "JupyterLite (Pyodide)" if IN_JUPYTERLITE else "lokales Jupyter")
print("numpy    :", np.__version__)
print("pandas   :", pd.__version__)
print("openpyxl :", openpyxl.__version__)
print("matplotlib:", mpl.__version__)

---
## 2. Versuchsaufbau und Pipettierschema

### 2.1 Belegung der 384-Well-Platte (Skript S. 28, Abbildung 7)

Jede Praktikumsgruppe belegt einen Block aus **6 Spalten**. Innerhalb eines Blocks gilt:

* **Spalten 1&ndash;3 des Blocks** &rarr; MC4R (ungerade Zeilen) bzw. mut. MC4R (gerade Zeilen)
* **Spalten 4&ndash;6 des Blocks** &rarr; pcDps (nur ungerade Zeilen)

Die Zeilen codieren die Stimulation (Abb. 8: eine 96-Well-Zeile bedient je zwei 384-Well-Zeilen):

| Zeilen | Stimulation |
|---|---|
| A / B | 10 &micro;M Forskolin (Positivkontrolle) |
| C / D | 0 M &alpha;-MSH (DMEM, unstimuliert) |
| E / F | 10<sup>&minus;11</sup> M &alpha;-MSH |
| G / H | 10<sup>&minus;10</sup> M &alpha;-MSH |
| I / J | 10<sup>&minus;9</sup> M &alpha;-MSH |
| K / L | 10<sup>&minus;8</sup> M &alpha;-MSH |
| M / N | 10<sup>&minus;7</sup> M &alpha;-MSH |
| O / P | 10<sup>&minus;6</sup> M &alpha;-MSH |

Dabei gilt: **ungerade Zeile** (A, C, E, G, I, K, M, O) = MC4R bzw. pcDps,
**gerade Zeile** (B, D, F, H, J, L, N, P) = mut. MC4R.

### 2.2 Unser Block

Unsere Messwerte stehen in **Spalte 13 bis Spalte 18, Zeile A bis Zeile P**:

* Spalten **13, 14, 15** = Triplikate MC4R / mut. MC4R
* Spalten **16, 17, 18** = Triplikate pcDps

In [ ]:
# ============================================================================
# KONFIGURATION  --  hier und nur hier muss bei einer anderen Gruppe
#                    bzw. einer anderen Messdatei etwas geaendert werden.
# ============================================================================

# Dateiname der Rohdaten (Tecan-Spark-Export)
DATEINAME = "1_3_6_7_ONE_Glo_Lumineszenz_Modified_20260910_145312_1.xlsx"

# --- Unser Plattenblock: Spalten 13-18, Zeilen A-P -------------------------
SPALTEN_REZEPTOR = [13, 14, 15]   # Triplikate MC4R (ungerade Zeilen) / mut. MC4R (gerade Zeilen)
SPALTEN_PCDPS    = [16, 17, 18]   # Triplikate pcDps (nur ungerade Zeilen)

# --- Zeilenpaare -> Stimulation (Skript S. 28, Abb. 7 und Abb. 8) ----------
#   (Zeile MC4R/pcDps, Zeile mut. MC4R): Bedingung
ZEILEN_SCHEMA = [
    ("A", "B", "FSK"),
    ("C", "D", "DMEM"),
    ("E", "F", -11),
    ("G", "H", -10),
    ("I", "J",  -9),
    ("K", "L",  -8),
    ("M", "N",  -7),
    ("O", "P",  -6),
]

# --- Konstrukte -----------------------------------------------------------
KONSTRUKTE = ["MC4R", "mut. MC4R", "pcDps"]

# Reihenfolge der Bedingungen in den Ergebnistabellen
BEDINGUNGEN = ["FSK", "DMEM", -11, -10, -9, -8, -7, -6]

# Die alpha-MSH-Konzentrationen der KWK (ohne Kontrollen)
KONZENTRATIONEN = [-11, -10, -9, -8, -7, -6]

# Position, an der der unstimulierte Wert (DMEM) in der Prism-Darstellung
# auf der log-Achse aufgetragen wird (vgl. Auswertungsvorlage: "-13")
X_DMEM = -13.0

# Bezugsgroesse der Normierung: unstimulierter Leervektor
REFERENZ_KONSTRUKT = "pcDps"
REFERENZ_BEDINGUNG = "DMEM"

# --- Farben (angelehnt an die Auswertungsvorlage) -------------------------
FARBEN_EXCEL = {"MC4R": "#9CC3E5", "mut. MC4R": "#F0A860", "pcDps": "#4F9A3D"}
FARBEN_PRISM = {"MC4R": "#5B9BD5", "mut. MC4R": "#C1C641", "pcDps": "#C9959B"}
FARBE_FSK, FARBE_DMEM = "#1F6E8C", "#E8833A"

print(f"Block  : Spalten {SPALTEN_REZEPTOR + SPALTEN_PCDPS}, Zeilen A-P")
print(f"Normierung auf: {REFERENZ_KONSTRUKT} / {REFERENZ_BEDINGUNG} (unstimulierter Leervektor)")

---
## 3. Rohdaten einlesen

Der Tecan-Spark-Export enthaelt zuerst einen langen Geraete-Kopf (Methode, Firmware,
Messparameter &hellip;) und erst danach das eigentliche Plattenraster. Dieses beginnt mit
einer Zeile, deren erste Zelle `<>` enthaelt; darunter folgen die Zeilen `A` bis `P`.

Die folgende Funktion sucht diesen Marker selbststaendig &ndash; sie funktioniert also
auch, wenn sich die Laenge des Kopfes bei einem anderen Export aendert.

### Hinweis zum Oeffnen der Datei in JupyterLite

> Ziehen Sie die Excel-Datei per Drag &amp; Drop in den **Dateibrowser links** in
> JupyterLite (am besten in denselben Ordner wie dieses Notebook oder in einen
> Unterordner `daten/`). Die naechste Zelle findet sie dann automatisch.

In [ ]:
def finde_messdatei(dateiname=DATEINAME, muster="*.xlsx"):
    """Sucht die Excel-Rohdatei an allen in JupyterLite / Jupyter ueblichen Orten.

    Rueckgabe: Path-Objekt der gefundenen Datei.
    """
    suchorte = [
        Path.cwd(), Path.cwd() / "daten", Path.cwd() / "data",
        Path.cwd().parent, Path.cwd().parent / "daten", Path.cwd().parent / "data",
        Path("/drive"), Path("/drive/daten"), Path("/drive/data"),   # JupyterLite-Laufwerk
    ]

    # 1) exakter Dateiname
    for ort in suchorte:
        kandidat = ort / dateiname
        try:
            if kandidat.is_file():
                return kandidat.resolve()
        except OSError:
            continue

    # 2) irgendeine passende .xlsx-Datei (z.B. nach Umbenennen beim Upload)
    gefunden = []
    for ort in suchorte:
        try:
            if ort.is_dir():
                gefunden += [p for p in sorted(ort.glob(muster)) if not p.name.startswith("~$")]
        except OSError:
            continue
    if gefunden:
        bevorzugt = [p for p in gefunden if "glo" in p.name.lower() or "lumin" in p.name.lower()]
        treffer = (bevorzugt or gefunden)[0]
        print(f"Hinweis: '{dateiname}' nicht gefunden - verwende stattdessen '{treffer.name}'.")
        return treffer.resolve()

    raise FileNotFoundError(
        f"Die Messdatei '{dateiname}' wurde nicht gefunden.\n"
        f"Durchsuchte Ordner: {[str(o) for o in suchorte]}\n\n"
        "-> Ziehen Sie die Excel-Datei per Drag & Drop in den Dateibrowser links "
        "(gleicher Ordner wie dieses Notebook oder Unterordner 'daten/') und fuehren "
        "Sie diese Zelle erneut aus."
    )


def lies_plattenraster(pfad):
    """Liest das 384-Well-Raster (Zeilen A-P, Spalten 1-24) aus dem Tecan-Export.

    Rueckgabe: DataFrame mit Index A..P und Spalten 1..24 (Lumineszenz in Counts/s).
    """
    blatt = openpyxl.load_workbook(pfad, data_only=True).worksheets[0]
    zellen = list(blatt.values)

    # Kopfzeile des Rasters suchen: erste Zelle enthaelt '<>'
    start = None
    for i, zeile in enumerate(zellen):
        if zeile and isinstance(zeile[0], str) and zeile[0].strip() == "<>":
            start = i
            break
    if start is None:
        raise ValueError("Plattenraster (Zeile mit '<>') im Excel-Export nicht gefunden.")

    # Spaltennummern; je nach Export stehen sie als Zahl oder als Text in der Datei
    spalten = []
    for wert in zellen[start][1:]:
        if wert is None or (isinstance(wert, str) and not wert.strip()):
            continue
        try:
            spalten.append(int(float(str(wert).strip())))
        except ValueError:
            continue

    daten = {}
    for zeile in zellen[start + 1:]:
        if not zeile or not isinstance(zeile[0], str):
            continue
        marke = zeile[0].strip()
        if not re.fullmatch(r"[A-P]", marke):      # Raster zu Ende
            break
        werte = []
        for v in zeile[1:1 + len(spalten)]:
            werte.append(np.nan if v in (None, "") else float(v))
        daten[marke] = werte

    raster = pd.DataFrame.from_dict(daten, orient="index", columns=spalten)
    raster.index.name = "Zeile"
    raster.columns.name = "Spalte"
    return raster


# --- ausfuehren -----------------------------------------------------------
PFAD = finde_messdatei()
print("Eingelesen:", PFAD)

platte = lies_plattenraster(PFAD)
print(f"Plattenformat: {platte.shape[0]} Zeilen x {platte.shape[1]} Spalten\n")
platte

### 3.1 Kontrolle: unser Block

Zur Sicherheit wird nur der Ausschnitt angezeigt, der uns gehoert (Spalten 13&ndash;18,
Zeilen A&ndash;P). Die leeren Zellen in den geraden Zeilen (B, D, F, &hellip;) sind korrekt:
dort wurde laut Pipettierschema kein pcDps ausgesaet.

In [ ]:
unser_block = platte.loc["A":"P", SPALTEN_REZEPTOR + SPALTEN_PCDPS]

print("Leere Zellen = laut Pipettierschema nicht belegt (pcDps nur in ungeraden Zeilen).\n")
unser_block

---
## 4. Wells den Konditionen zuordnen

Aus dem Plattenraster wird eine &bdquo;lange&ldquo; Tabelle (*tidy data*) erzeugt: eine Zeile
pro Well, mit Konstrukt, Bedingung, Well-Koordinate und Messwert. Das macht alle
folgenden Rechenschritte nachvollziehbar und leicht ueberpruefbar.

In [ ]:
def baue_messtabelle(raster):
    """Ordnet jedem Well des Blocks Konstrukt und Bedingung zu (Pipettierschema Abb. 7)."""
    eintraege = []

    for zeile_wt, zeile_mut, bedingung in ZEILEN_SCHEMA:

        # --- MC4R (wildtypisch): ungerade Zeile, Spalten 13-15 ---
        for spalte in SPALTEN_REZEPTOR:
            eintraege.append({
                "Konstrukt": "MC4R", "Bedingung": bedingung,
                "Well": f"{zeile_wt}{spalte}",
                "Lumineszenz": raster.at[zeile_wt, spalte],
            })

        # --- mut. MC4R: gerade Zeile, Spalten 13-15 ---
        for spalte in SPALTEN_REZEPTOR:
            eintraege.append({
                "Konstrukt": "mut. MC4R", "Bedingung": bedingung,
                "Well": f"{zeile_mut}{spalte}",
                "Lumineszenz": raster.at[zeile_mut, spalte],
            })

        # --- pcDps (Leervektor): ungerade Zeile, Spalten 16-18 ---
        for spalte in SPALTEN_PCDPS:
            eintraege.append({
                "Konstrukt": "pcDps", "Bedingung": bedingung,
                "Well": f"{zeile_wt}{spalte}",
                "Lumineszenz": raster.at[zeile_wt, spalte],
            })

    tabelle = pd.DataFrame(eintraege)
    tabelle["Konstrukt"] = pd.Categorical(tabelle["Konstrukt"], KONSTRUKTE, ordered=True)
    return tabelle


messwerte = baue_messtabelle(platte)

print(f"{len(messwerte)} Wells zugeordnet "
      f"({messwerte['Lumineszenz'].isna().sum()} davon ohne Messwert)\n")
messwerte.head(12)

### 4.1 Rohdaten als Uebersichtstabelle

Dieselben Daten noch einmal im Format der Auswertungsvorlage: pro Kondition und
Bedingung die drei Triplikate nebeneinander.

In [ ]:
triplikate = (messwerte
              .assign(Replikat=messwerte.groupby(["Konstrukt", "Bedingung"], observed=True).cumcount() + 1)
              .pivot_table(index=["Konstrukt", "Bedingung"], columns="Replikat",
                           values="Lumineszenz", observed=True, sort=False))
triplikate.columns = [f"Triplikat {i}" for i in triplikate.columns]
triplikate = triplikate.reindex(
    pd.MultiIndex.from_product([KONSTRUKTE, BEDINGUNGEN], names=["Konstrukt", "Bedingung"])
).dropna(how="all")

triplikate

---
## 5. Mittelwerte und Streuung der Triplikate

Fuer jede Kombination aus Konstrukt und Bedingung werden Mittelwert und
Standardabweichung der drei technischen Replikate berechnet.

> Die Standardabweichung beschreibt hier **nur die Pipettier-/Messstreuung innerhalb
> eines einzigen Experiments** (technische Replikate). Da der Versuch nur einmal
> durchgefuehrt wurde, wird sie laut Praktikumsskript in den Abbildungen **nicht** als
> Fehlerbalken dargestellt &ndash; sie dient ausschliesslich der Qualitaetsbeurteilung.

In [ ]:
kennwerte = (messwerte
             .groupby(["Konstrukt", "Bedingung"], observed=True)["Lumineszenz"]
             .agg(Mittelwert="mean",
                  SD=lambda s: s.std(ddof=1),     # Stichproben-SD, wie in Excel STABW.S
                  n="count")
             .reset_index())

# Variationskoeffizient als Mass fuer die Streuung der Triplikate
kennwerte["VK [%]"] = 100 * kennwerte["SD"] / kennwerte["Mittelwert"]

kennwerte = (kennwerte
             .set_index(["Konstrukt", "Bedingung"])
             .reindex(pd.MultiIndex.from_product([KONSTRUKTE, BEDINGUNGEN],
                                                 names=["Konstrukt", "Bedingung"])))
kennwerte

### 5.1 Auffaellige Triplikate

Ein einzelner stark abweichender Well verschiebt den Mittelwert &ndash; und damit, falls er
im Referenz-Well liegt, die gesamte Normierung. Die folgende Zelle listet deshalb alle
Triplikate mit einem Variationskoeffizienten &gt; 50 % auf und zeigt, wie sich der
Mittelwert ohne den am staerksten abweichenden Einzelwert veraendern wuerde.

> Die Werte werden **nicht automatisch entfernt**. Ob ein Ausreisser gestrichen wird,
> ist eine fachliche Entscheidung, die begruendet und im Protokoll dokumentiert werden
> muss.

In [ ]:
GRENZE_VK = 50.0   # Prozent

auffaellig = kennwerte[kennwerte["VK [%]"] > GRENZE_VK]

if auffaellig.empty:
    print(f"Keine Triplikate mit VK > {GRENZE_VK:.0f} % - Streuung unauffaellig.")
else:
    print(f"Triplikate mit VK > {GRENZE_VK:.0f} %:\n")
    for (konstrukt, bedingung) in auffaellig.index:
        gruppe = messwerte[(messwerte["Konstrukt"] == konstrukt) &
                           (messwerte["Bedingung"] == bedingung)]
        werte = gruppe["Lumineszenz"].to_numpy(float)
        wells = gruppe["Well"].tolist()

        # am staerksten vom Median abweichender Einzelwert
        idx_weg = int(np.argmax(np.abs(werte - np.median(werte))))
        ohne = np.delete(werte, idx_weg)

        print(f"  {konstrukt} / {bedingung}")
        paare = ", ".join(f"{w} = {int(v)}" for w, v in zip(wells, werte))
        print(f"     Wells        : {paare}")
        print(f"     Mittelwert   : {werte.mean():8.1f}  (VK {auffaellig.loc[(konstrukt, bedingung), 'VK [%]']:.0f} %)")
        print(f"     ohne {wells[idx_weg]:<5}   : {ohne.mean():8.1f}  "
              f"(= {ohne.mean() / werte.mean():.2f}-fach des urspruenglichen Werts)\n")

---
## 6. Normierung auf den unstimulierten Leervektor

Die absoluten Lumineszenz-Werte haengen von Zellzahl, Transfektionseffizienz und
Substratmenge ab und sind zwischen Platten nicht vergleichbar. Deshalb wird jeder
Mittelwert auf den Mittelwert des **unstimulierten Leervektors** (pcDps, 0 M &alpha;-MSH)
bezogen:

$$
\textsf{x-fold of pcDps unstimulated}
\;=\;
\frac{\overline{\textsf{RLU}}_{\textsf{Konstrukt, Bedingung}}}
     {\overline{\textsf{RLU}}_{\textsf{pcDps, 0 M}}}
$$

Damit ist pcDps / 0 M per Definition gleich 1. Alle Werte geben an, um welchen Faktor
das jeweilige Signal ueber (bzw. unter) der basalen Aktivitaet des Leervektors liegt.

In [ ]:
# --- Referenzwert: pcDps, unstimuliert -----------------------------------
referenz = kennwerte.loc[(REFERENZ_KONSTRUKT, REFERENZ_BEDINGUNG), "Mittelwert"]

print(f"Referenz (Mittelwert {REFERENZ_KONSTRUKT}, {REFERENZ_BEDINGUNG}): "
      f"{referenz:,.2f} Counts/s")

if not np.isfinite(referenz) or referenz <= 0:
    raise ValueError("Referenzwert ist 0 oder fehlt - Normierung nicht moeglich.")

# --- Normierung ----------------------------------------------------------
kennwerte["x-fold"]    = kennwerte["Mittelwert"] / referenz
kennwerte["SD x-fold"] = kennwerte["SD"]         / referenz

kennwerte[["Mittelwert", "SD", "n", "VK [%]", "x-fold", "SD x-fold"]]

### 6.1 Ergebnistabelle

Kompakte Darstellung der normierten Werte (Aufbau wie in der Auswertungsvorlage):
Zeilen = Konstrukte, Spalten = Stimulation.

In [ ]:
ergebnis = (kennwerte["x-fold"]
            .unstack("Bedingung")
            .reindex(index=KONSTRUKTE, columns=BEDINGUNGEN))
ergebnis.columns = ["FSK 10 µM", "DMEM (0 M)"] + [f"10^{c} M" for c in KONZENTRATIONEN]
ergebnis.index.name = "x-fold of pcDps unstimulated"

ergebnis.round(4)

---
## 7. Abbildung 1 &ndash; Konzentrations-Wirkungs-Kurve

Auftragung der normierten Signale gegen die &alpha;-MSH-Konzentration. Die x-Achse ist
kategorial (DMEM, dann 10<sup>&minus;11</sup> &hellip; 10<sup>&minus;6</sup> M), wie in der
Auswertungsvorlage. Ohne Fehlerbalken, da nur ein Experiment vorliegt.

In [ ]:
# --- Daten fuer die KWK (DMEM + alle alpha-MSH-Konzentrationen) -----------
kwk_bedingungen = ["DMEM"] + KONZENTRATIONEN
kwk = ergebnis.copy()
kwk.columns = BEDINGUNGEN
kwk = kwk[kwk_bedingungen]

x_kat  = np.arange(len(kwk_bedingungen))
labels = ["DMEM"] + [str(c) for c in KONZENTRATIONEN]

fig, ax = plt.subplots(figsize=(6.2, 4.0))

for konstrukt in KONSTRUKTE:
    ax.plot(x_kat, kwk.loc[konstrukt].values,
            marker="o", markersize=6, linewidth=2,
            color=FARBEN_EXCEL[konstrukt], label=konstrukt,
            markeredgecolor="white", markeredgewidth=0.6)

ax.set_title("reporter gene assay", fontsize=12, pad=12)
ax.set_xlabel(r"$\alpha$-MSH, FSK, log [M]", labelpad=8)
ax.set_ylabel("x-fold of pcDps unstimulated", labelpad=8)
ax.set_xticks(x_kat)
ax.set_xticklabels(labels)
ax.set_xlim(-0.4, len(x_kat) - 0.6)
ax.set_ylim(bottom=0)

ax.grid(axis="y", color="#D9D9D9", linewidth=0.7)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18),
          ncol=3, frameon=False, handlelength=2.2)

fig.savefig("Abb1_Konzentrations-Wirkungs-Kurve.png")
plt.show()

---
## 8. Abbildung 2 &ndash; Kontrollen (Forskolin vs. DMEM)

Forskolin aktiviert die Adenylylcyclase **direkt**, also unabhaengig vom Rezeptor.
Die FSK-Kontrolle zeigt daher, ob Zellen, Transfektion, Reportergen und Substrat
ueberhaupt funktioniert haben. DMEM ist die zugehoerige unstimulierte Kontrolle.

In [ ]:
fsk  = [ergebnis.loc[k].iloc[0] for k in KONSTRUKTE]   # Spalte "FSK 10 µM"
dmem = [ergebnis.loc[k].iloc[1] for k in KONSTRUKTE]   # Spalte "DMEM (0 M)"

x     = np.arange(len(KONSTRUKTE))
breit = 0.35

fig, ax = plt.subplots(figsize=(5.6, 4.0))

ax.bar(x - breit / 2, fsk,  breit, label="FSK 10 µM", color=FARBE_FSK)
ax.bar(x + breit / 2, dmem, breit, label="DMEM",      color=FARBE_DMEM)

ax.axhline(1.0, color="#7F7F7F", linewidth=1.0, linestyle=":")

ax.set_title("FSK control", fontsize=12, pad=12)
ax.set_ylabel("x-fold of pcDps unstimulated", labelpad=8)
ax.set_xticks(x)
ax.set_xticklabels(KONSTRUKTE)
ax.set_ylim(bottom=0)

ax.grid(axis="y", color="#D9D9D9", linewidth=0.7)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12),
          ncol=2, frameon=False)

fig.savefig("Abb2_FSK-Kontrolle.png")
plt.show()

---
## 9. Kurvenanpassung und EC<sub>50</sub>

Angepasst wird das in der Pharmakologie uebliche Modell
*log(agonist) vs. response* mit **drei Parametern** (variable Steigung fixiert auf 1,
Hill-Slope = 1):

$$
Y \;=\; \textsf{Bottom} \;+\; \frac{\textsf{Top} - \textsf{Bottom}}{1 + 10^{\,(\log EC_{50} \,-\, X)}}
$$

mit $X = \log_{10}(c_{\alpha\text{-MSH}} \,/\, \mathrm{M})$.

* **Bottom** &ndash; basales Signal ohne Agonist
* **Top** &ndash; maximales Signal bei saettigender Agonistkonzentration
* **EC<sub>50</sub>** &ndash; Konzentration, bei der das halbmaximale Signal erreicht wird
  (Mass fuer die Potenz des Agonisten am jeweiligen Rezeptor)

Wie in der Auswertungsvorlage geht der unstimulierte Wert (DMEM) als Punkt bei
$X = -13$ in den Fit ein; die Forskolin-Kontrolle bleibt aussen vor, da sie den
Rezeptor umgeht.

In [ ]:
def dosis_wirkung(x, bottom, top, log_ec50):
    """log(agonist) vs. response, 3 Parameter (Hill-Slope = 1)."""
    return bottom + (top - bottom) / (1.0 + 10.0 ** (log_ec50 - x))


def fitte_kurve(x, y):
    """Passt das 3-Parameter-Modell an und liefert Parameter + Guetemasse."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    gueltig = np.isfinite(x) & np.isfinite(y)
    x, y = x[gueltig], y[gueltig]

    ergebnis = {"Bottom": np.nan, "Top": np.nan, "LogEC50": np.nan, "EC50 [M]": np.nan,
                "Span": np.nan, "R²": np.nan, "n Punkte": len(x), "Status": ""}
    if len(x) < 4:
        ergebnis["Status"] = "zu wenige Punkte"
        return ergebnis

    start  = [float(np.min(y)), float(np.max(y)), float(np.median(x))]
    grenze = ([-np.inf, -np.inf, x.min() - 3], [np.inf, np.inf, x.max() + 3])

    try:
        popt, _ = curve_fit(dosis_wirkung, x, y, p0=start, bounds=grenze, maxfev=20000)
    except Exception as fehler:
        ergebnis["Status"] = f"Fit fehlgeschlagen ({type(fehler).__name__})"
        return ergebnis

    bottom, top, log_ec50 = popt
    rest  = y - dosis_wirkung(x, *popt)
    sq_r  = float(np.sum(rest ** 2))
    sq_t  = float(np.sum((y - y.mean()) ** 2))
    r2    = 1 - sq_r / sq_t if sq_t > 0 else np.nan

    ergebnis.update({
        "Bottom": bottom, "Top": top, "LogEC50": log_ec50,
        "EC50 [M]": 10.0 ** log_ec50, "Span": top - bottom, "R²": r2,
    })

    # Plausibilitaetspruefung: liegt die EC50 im gemessenen Bereich, gibt es ueberhaupt ein Fenster?
    hinweise = []
    if not (x.min() <= log_ec50 <= x.max()):
        hinweise.append("EC50 ausserhalb des gemessenen Bereichs")
    if abs(top - bottom) < 0.2 * max(abs(bottom), 1e-9):
        hinweise.append("kein nennenswertes Signalfenster (Top ≈ Bottom)")
    if np.isfinite(r2) and r2 < 0.8:
        hinweise.append("schlechte Anpassung (R² < 0,8)")
    ergebnis["Status"] = "; ".join(hinweise) if hinweise else "ok"
    return ergebnis


# --- x-Werte: DMEM bei -13, danach die echten Konzentrationen -------------
x_fit = np.array([X_DMEM] + [float(c) for c in KONZENTRATIONEN])

fit_ergebnisse = {}
for konstrukt in KONSTRUKTE:
    y = kennwerte.loc[konstrukt].loc[["DMEM"] + KONZENTRATIONEN, "x-fold"].to_numpy(float)
    fit_ergebnisse[konstrukt] = fitte_kurve(x_fit, y)

fits = pd.DataFrame(fit_ergebnisse).T
fits.index.name = "Nonlin fit"
fits

### 9.1 Ergebnisse der Kurvenanpassung

Darstellung im Format der Auswertungsvorlage (*Table of results*). Die Spalte
**Status** weist automatisch auf Fits hin, die rechnerisch zwar ein Ergebnis liefern,
inhaltlich aber nicht belastbar sind.

In [ ]:
fits_anzeige = pd.DataFrame({
    "Bottom":   fits["Bottom"].map(lambda v: f"{v:.4g}"),
    "Top":      fits["Top"].map(lambda v: f"{v:.4g}"),
    "Span":     fits["Span"].map(lambda v: f"{v:.4g}"),
    "LogEC50":  fits["LogEC50"].map(lambda v: f"{v:.4g}"),
    "EC50 [M]": fits["EC50 [M]"].map(lambda v: f"{v:.4g}" if np.isfinite(v) else "-"),
    "EC50 [nM]": fits["EC50 [M]"].map(lambda v: f"{v * 1e9:.4g}" if np.isfinite(v) else "-"),
    "R²":       fits["R²"].map(lambda v: f"{v:.4f}"),
    "n Punkte": fits["n Punkte"],
    "Status":   fits["Status"],
})
fits_anzeige.index.name = "Nonlin fit"
fits_anzeige

In [ ]:
# --- EC50 in gut lesbarer Form -------------------------------------------
print("EC50-Werte (log(agonist) vs. response, 3 Parameter)")
print("-" * 62)
for konstrukt in KONSTRUKTE:
    e = fit_ergebnisse[konstrukt]
    ec50 = e["EC50 [M]"]
    if np.isfinite(ec50):
        print(f"{konstrukt:<12}  EC50 = {ec50:.3e} M  = {ec50 * 1e9:,.3f} nM"
              f"   |  R² = {e['R²']:.4f}   |  {e['Status']}")
    else:
        print(f"{konstrukt:<12}  EC50 = nicht bestimmbar   |  {e['Status']}")
print("-" * 62)

verhaeltnis = fit_ergebnisse["mut. MC4R"]["EC50 [M]"] / fit_ergebnisse["MC4R"]["EC50 [M]"]
if np.isfinite(verhaeltnis):
    print(f"\nEC50(mut. MC4R) / EC50(MC4R) = {verhaeltnis:,.2f}"
          f"   ->  {'Rechtsverschiebung (geringere Potenz)' if verhaeltnis > 1 else 'Linksverschiebung (hoehere Potenz)'}"
          " der Mutante")

---
## 10. Abbildung 3 &ndash; Gesamtdarstellung (Auswertungsvorlage)

Zusammenfassende Abbildung im Stil der Vorlage: Messpunkte (Kreise) mit den
angepassten Kurven (gestrichelt), der unstimulierte Wert bei log[M] = &minus;13 und die
Forskolin-Positivkontrolle als Quadrate links neben der Konzentrationsachse
(&bdquo;pos ctrl&ldquo;).

In [ ]:
X_POSCTRL = -14.6          # Position der Positivkontrolle links der Achse

fig, ax = plt.subplots(figsize=(6.8, 4.8))

x_punkte = np.array([X_DMEM] + [float(c) for c in KONZENTRATIONEN])
x_glatt  = np.linspace(X_DMEM, KONZENTRATIONEN[-1], 400)

for konstrukt in KONSTRUKTE:
    farbe = FARBEN_PRISM[konstrukt]
    y = kennwerte.loc[konstrukt].loc[["DMEM"] + KONZENTRATIONEN, "x-fold"].to_numpy(float)

    # Messpunkte
    ax.plot(x_punkte, y, linestyle="none", marker="o", markersize=7,
            color=farbe, label=konstrukt, zorder=3)

    # angepasste Kurve
    e = fit_ergebnisse[konstrukt]
    if np.isfinite(e["LogEC50"]):
        ax.plot(x_glatt, dosis_wirkung(x_glatt, e["Bottom"], e["Top"], e["LogEC50"]),
                linestyle="--", linewidth=1.5, color=farbe, zorder=2)

    # Positivkontrolle (10 µM Forskolin) als Quadrat
    ax.plot(X_POSCTRL, kennwerte.loc[(konstrukt, "FSK"), "x-fold"],
            linestyle="none", marker="s", markersize=7, color=farbe, zorder=3)

# --- Achsen im Stil der Vorlage ------------------------------------------
ticks  = [X_POSCTRL] + list(range(-13, -5))
labels = ["pos\nctrl"] + [str(t) for t in range(-13, -5)]
ax.set_xticks(ticks)
ax.set_xticklabels(labels)
ax.set_xlim(X_POSCTRL - 0.8, -5.4)

ax.set_xlabel(r"$\alpha$-MSH, FSK, log [M]", fontweight="bold", labelpad=8)
ax.set_ylabel("x-fold of pcDps unstimulated", fontweight="bold", labelpad=8)
ax.set_title("Reporter gene assay G$_\\mathrm{s}$", fontweight="bold", pad=14)

obergrenze = float(np.nanmax(kennwerte["x-fold"].to_numpy(float)))
ax.set_ylim(0, obergrenze * 1.25)

# Achsen leicht abgesetzt (Prism-Optik)
ax.spines["left"].set_position(("outward", 8))
ax.spines["bottom"].set_position(("outward", 8))
ax.tick_params(direction="out", length=5, width=1.0)

# optische Trennung der Positivkontrolle von der Konzentrationsachse
ax.axvline(X_POSCTRL + 0.75, color="#BFBFBF", linewidth=0.8, linestyle=":")

griffe = [plt.Line2D([], [], linestyle="none", marker="o", markersize=7,
                     color=FARBEN_PRISM[k], label=k) for k in KONSTRUKTE]
griffe.append(plt.Line2D([], [], linestyle="none", marker="s", markersize=7,
                         color="#7F7F7F", label="FSK 10 µM (pos ctrl)"))
griffe.append(plt.Line2D([], [], linestyle="--", color="#7F7F7F",
                         label="Fit: log(agonist) vs. response"))
ax.legend(handles=griffe, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

fig.savefig("Abb3_Reporter-gene-assay-Gs.png")
plt.show()

---
## 11. Ergebnisse sichern

Alle Tabellen werden als CSV-Dateien abgelegt (in JupyterLite erscheinen sie im
Dateibrowser links und koennen von dort heruntergeladen werden). Die Abbildungen
wurden bereits als PNG (300 dpi) gespeichert.

In [ ]:
export = {
    "Ergebnis_01_Rohdaten_Triplikate.csv": triplikate,
    "Ergebnis_02_Mittelwerte_SD.csv":      kennwerte,
    "Ergebnis_03_Normiert_x-fold.csv":     ergebnis,
    "Ergebnis_04_Kurvenanpassung.csv":     fits,
}

for name, tabelle in export.items():
    tabelle.to_csv(name, sep=";", decimal=",", encoding="utf-8-sig")
    print("gespeichert:", name)

for name in ["Abb1_Konzentrations-Wirkungs-Kurve.png",
             "Abb2_FSK-Kontrolle.png",
             "Abb3_Reporter-gene-assay-Gs.png"]:
    print("gespeichert:", name)

---
## 12. Qualitaetskontrolle und Interpretation

Bevor die Zahlen biologisch interpretiert werden, muss geprueft werden, ob der Assay
ueberhaupt funktioniert hat. Die folgende Zelle prueft das automatisch anhand von
drei Kriterien:

1. **Signalhoehe** &ndash; liegt die Lumineszenz deutlich ueber dem Geraeterauschen?
2. **Forskolin-Kontrolle** &ndash; FSK umgeht den Rezeptor und muss in *allen* drei
   Konditionen (auch im Leervektor!) ein deutlich erhoehtes Signal erzeugen. Tut es das
   nicht, sind Zellen, Transfektion, Reportergen oder Substrat das Problem &ndash; nicht
   der Rezeptor.
3. **Signalfenster** &ndash; gibt es ueberhaupt einen Unterschied zwischen unstimuliert
   und maximal stimuliert?

In [ ]:
print("=" * 70)
print("QUALITAETSKONTROLLE")
print("=" * 70)

# --- 1) Signalhoehe ------------------------------------------------------
max_signal = float(np.nanmax(kennwerte["Mittelwert"].to_numpy(float)))
print(f"\n1) Signalhoehe")
print(f"   hoechster Mittelwert im Block : {max_signal:,.0f} Counts/s")
print(f"   Referenz (pcDps unstimuliert) : {referenz:,.0f} Counts/s")
if max_signal < 1000:
    print("   ! Alle Werte liegen im Bereich des Geraeterauschens (< 1.000 Counts/s).")
else:
    print("   Signal deutlich ueber dem Rauschen.")

# --- 2) Forskolin-Kontrolle ---------------------------------------------
print(f"\n2) Forskolin-Kontrolle (muss in allen Konditionen >> 1 sein)")
fsk_ok = True
for konstrukt in KONSTRUKTE:
    wert = kennwerte.loc[(konstrukt, "FSK"), "x-fold"]
    basis = kennwerte.loc[(konstrukt, "DMEM"), "x-fold"]
    verh  = wert / basis if basis else np.nan
    status = "ok" if verh >= 2 else "! zu gering"
    fsk_ok &= verh >= 2
    print(f"   {konstrukt:<12} FSK = {wert:6.2f} x-fold   "
          f"(FSK/DMEM = {verh:5.2f})   {status}")

# --- 3) Signalfenster ----------------------------------------------------
print(f"\n3) Signalfenster der KWK (maximal stimuliert / unstimuliert)")
for konstrukt in KONSTRUKTE:
    basis = kennwerte.loc[(konstrukt, "DMEM"), "x-fold"]
    top   = float(np.nanmax(kennwerte.loc[konstrukt].loc[KONZENTRATIONEN, "x-fold"].to_numpy(float)))
    print(f"   {konstrukt:<12} {top / basis:5.2f}-fach" if basis else f"   {konstrukt:<12}   -")

# --- 4) Streuung der Triplikate -----------------------------------------
vk_median = float(np.nanmedian(kennwerte["VK [%]"].to_numpy(float)))
print(f"\n4) Streuung der Triplikate: Median-VK = {vk_median:.1f} %")
if vk_median > 20:
    print("   ! Hohe technische Streuung (VK > 20 %).")

# --- Gesamtbewertung -----------------------------------------------------
print("\n" + "=" * 70)
if max_signal < 1000 or not fsk_ok:
    print("GESAMTBEWERTUNG: Der Assay hat in diesem Block NICHT funktioniert.")
    print("Die EC50-Werte aus Abschnitt 9 sind damit NICHT interpretierbar.")
    print("Moegliche Ursachen: fehlgeschlagene Transfektion, Zellverlust beim")
    print("Mediumwechsel, Substrat- oder Pipettierfehler.")
else:
    print("GESAMTBEWERTUNG: Assay plausibel - EC50-Werte koennen interpretiert werden.")
print("=" * 70)

---
## 13. Zusammenfassung

**Rechenweg**

1. Rohdaten (Counts/s) aus dem Tecan-Export, Block Spalten 13&ndash;18 / Zeilen A&ndash;P
2. Zuordnung ueber das Pipettierschema (Skript S. 28, Abb. 7 und 8)
3. Mittelwert der Triplikate je Konstrukt und Bedingung
4. Normierung: Mittelwert / Mittelwert(pcDps, 0 M) = *x-fold of pcDps unstimulated*
5. Fit `Y = Bottom + (Top - Bottom) / (1 + 10^(LogEC50 - X))` an DMEM (X = &minus;13)
   und 10<sup>&minus;11</sup> &hellip; 10<sup>&minus;6</sup> M &rarr; EC<sub>50</sub>

**Erwartetes Ergebnis (Literatur / Vorlage)**

* MC4R (wt): konzentrationsabhaengiger Signalanstieg, EC<sub>50</sub> im niedrigen nanomolaren Bereich
* mut. MC4R (Ile194Phe): reduziertes Maximalsignal und/oder Rechtsverschiebung der Kurve
  &rarr; Funktionsverlust, passend zum klinischen Bild der monogenen Adipositas
* pcDps: keine Konzentrationsabhaengigkeit (flach bei ca. 1)
* FSK: deutliche Erhoehung in **allen** Konditionen, da Forskolin die Adenylylcyclase
  direkt aktiviert

Ob die eigenen Daten dieses Muster zeigen, beantwortet die Qualitaetskontrolle in
Abschnitt 12.